# Backtest of the live strategy

One notebook for the strategy backtest (replaces `strategy_backtest_v2/v3/v4.ipynb`). It uses the same engine as the daily
pipeline (`backtest_engine.py`), so the backtest trades exactly what the live rules would have traded.

- **Rules:** `backtest_engine.WINNER` (the live rules; the tag is printed below).
- **Period:** walk-forward from 2022-04-01 to the last cached bar; decisions at the close, fills at the next open, 0.1% cost per side.
  The *never-seen* part (2022-04 → 2024-09) was not used when the rules were chosen, so it is the most honest number.
- **Data:** the cached daily bars in `Reports/cache/bars_daily_long.pkl` (no API quota is used). The stock list is today's
  universe, picked with hindsight, so expect live results to be weaker than the backtest.
- **Outputs:** `Reports/backtest_summary.csv` (the tables below; shown in the app under Details → Backtest) and
  `Reports/backtest_per_stock.csv` (per-stock results; shown in the app's "More about" box).
- Run it with *Run All*, or `python run_all.py --backtests`.

## 1. Settings
`VARIANT` = the rule keys to change for the comparison in section 5 (any key of `backtest_engine.WINNER`). Examples:
`{"midweek_exit_below": None}` (no rank-30 exit), `{"max_pick_rank": None, "cap_soft": False}` (picks from any rank, hard
sector limit), `{"midweek_swap": None}` (weekly only), `{"n": 8}` (hold 8 stocks).

In [1]:
import pandas as pd

import backtest_engine as be

REFRESH_BARS = False                   # True = download the bars again (Alpaca market data; changes the pinned test numbers)
VARIANT = {"earnings_block_days": None}   # rule keys to change for section 5
SAVE = True                            # write Reports/backtest_summary.csv and Reports/backtest_per_stock.csv
pd.set_option("display.float_format", "{:.2f}".format)
print("Live rules:", be.WINNER["tag"], "-", be.WINNER["name"])

Live rules: C6-U96-T20-MW30-E5 - C6: weekly top-10 ranking, max 4 per sector + soft QQQ regime (expanded universe, 96 stocks) [picks from ranks 1-20 only; sector cap relaxed to fill the 10 slots; top-3 swaps ignore the cap] + mid-week swap (Mon/Wed close: top 3 in, below rank 15 out) + mid-week exit (sell if worse than rank 30, cash until Friday) + no new buys with earnings in the next 5 days


## 2. Load prices and scores

In [2]:
inp = be.backtest_inputs(refresh=REFRESH_BARS)
dates = inp["close"].index
print(f"{len(be.TRADABLE)} stocks, bars {dates[0]:%Y-%m-%d} → {dates[-1]:%Y-%m-%d}; backtest from {be.WALK_FORWARD_START}")

96 stocks, bars 2020-06-01 → 2026-09-24; backtest from 2022-04-01


## 3. Live rules vs QQQ and SPY
Total return, yearly growth (CAGR), Sharpe (return per unit of risk; higher is better) and the worst drop (max DD) for each period.

In [3]:
live = be.run_rules(inp)
summary = be.period_rows(be.WINNER["tag"], live["res"]["equity"])
for bench in ("QQQ", "SPY"):
    summary += be.period_rows(f"{bench} buy & hold", be.buy_and_hold(inp, bench)["equity"])
summary = pd.DataFrame(summary)
stats = be.trade_stats(live["res"], dates)
summary.loc[(summary["Strategy"] == be.WINNER["tag"]) & (summary["Period"] == "Walk-forward"), list(stats)] = list(stats.values())
summary.pivot_table(index="Period", columns="Strategy", values=["Total Return %", "Sharpe", "Max DD %"], sort=False)

Total Return %                                \
Strategy                     C6-U96-T20-MW30-E5 QQQ buy & hold SPY buy & hold   
Period                                                                          
Walk-forward                             399.90         110.10          79.58   
Never-seen 2022-04 → 2024-09              33.20          32.63          28.38   
Last 2 years                             257.33          54.23          37.39   
Last 1 year                               52.82          24.91          17.30   

                                         Sharpe                                \
Strategy                     C6-U96-T20-MW30-E5 QQQ buy & hold SPY buy & hold   
Period                                                                          
Walk-forward                               1.35           0.85           0.85   
Never-seen 2022-04 → 2024-09               0.60           0.61           0.67   
Last 2 years                               1.97           1.11           1.05   
Last 1 year                                1.39           1.22           1.30   

                                       Max DD %                                
Strategy                     C6-U96-T20-MW30-E5 QQQ buy & hold SPY buy & hold  
Period                                                                         
Walk-forward                             -30.76         -29.07         -21.29  
Never-seen 2022-04 → 2024-09             -29.33         -29.07         -21.29  
Last 2 years                             -30.76         -22.77         -18.75  
Last 1 year                              -23.53         -11.96          -8.88

In [4]:
print(f"Trades: {stats['Trades']} · win rate {stats['Win rate %']:.0f}% · median trade {stats['Median trade %']:+.2f}% · "
      f"median hold {stats['Median hold (sessions)']:.0f} sessions (medians, not averages)")

Trades: 1021 · win rate 45% · median trade -0.74% · median hold 6 sessions (medians, not averages)


**Earnings rule (`earnings_block_days`):** no new buy when the next earnings date is within that many calendar days
after the decision date. The backtest can only use the earnings dates on disk (`Reports/earnings_date.csv`), which start in
late 2024 and miss some stocks, so for the earnings rule this backtest is **partial** and for information only. Section 5
compares with the rule switched off (`VARIANT = {"earnings_block_days": None}`).

In [5]:
earn = be.load_earnings()
have = sorted(set(earn["Symbol"]) & set(be.TRADABLE))
missing = sorted(set(be.TRADABLE) - set(have))
if be.WINNER.get("earnings_block_days"):
    print(f"Earnings rule on ({be.WINNER['earnings_block_days']} days). Dates on file for {len(have)} of {len(be.TRADABLE)} stocks, "
          f"{earn['Earnings Date'].min():%Y-%m-%d} → {earn['Earnings Date'].max():%Y-%m-%d}; before that, and for the "
          f"{len(missing)} stocks without dates, the rule cannot act. PARTIAL test.")
    print("No dates:", " ".join(missing) or "none")
else:
    print("Earnings rule off (earnings_block_days = None).")

Earnings rule on (5 days). Dates on file for 78 of 96 stocks, 2024-09-25 → 2026-12-09; before that, and for the 18 stocks without dates, the rule cannot act. PARTIAL test.
No dates: APA BA BE C CLS COF CRDO DVN FANG FCX LITE LYB NBIS OXY PH RBRK TRGP URI


## 4. Per stock
For each stock: how often the rules traded it and how those trades went, next to simply holding the stock over the same
period. The line below gives the median over all stocks.

In [6]:
per_stock = be.per_stock_table(live, inp)
med = per_stock.median(numeric_only=True)
print(f"Median stock: {med['Closed trades']:.0f} closed trades, win rate {med['Win rate %']:.0f}%, median trade "
      f"{med['Median trade %']:+.2f}%, held {med['Held % of sessions']:.0f}% of sessions; buy & hold {med['Buy & hold %']:+.0f}%")
traded = per_stock[per_stock["Closed trades"] >= 3].sort_values("Median trade %")
pd.concat([traded.tail(5), traded.head(5)])[["Symbol", "Sector", "Closed trades", "Win rate %", "Median trade %",
                                             "Median hold (sessions)", "Buy & hold %"]]

Median stock: 11 closed trades, win rate 47%, median trade -0.92%, held 10% of sessions; buy & hold +95%


,Symbol,Sector,Closed trades,Win rate %,Median trade %,Median hold (sessions),Buy & hold %
66,CVNA,Consumer Discretionary,10,70.00,5.24,19.00,167.75
73,RKLB,Industrials,14,64.29,8.44,13.00,814.41
93,LITE,Technology,6,66.67,10.03,10.00,850.72
26,IREN,Energy,7,71.43,12.21,5.00,192.83
69,MARA,Technology,3,66.67,14.10,14.00,-53.63
53,UMAC,Technology,4,25.00,-17.63,5.00,498.25
65,CIFR,Technology,13,30.77,-9.29,12.00,395.89
62,APLD,Technology,14,35.71,-8.20,5.00,475.74
60,ACHR,Industrials,12,25.00,-7.84,5.00,17.98
91,CRDO,Technology,20,35.00,-7.13,5.00,1186.74


## 5. Compare a rule variant
Runs the same backtest with the keys in `VARIANT` changed (section 1) and shows both side by side. Nothing in the live
rules changes; to adopt a variant, edit `WINNER` in `backtest_engine.py` and run `python run_all.py`.

In [7]:
variant_name = "Variant: " + ", ".join(f"{k}={v}" for k, v in VARIANT.items())
variant = be.run_rules(inp, **VARIANT)
vrows = pd.DataFrame(be.period_rows(variant_name, variant["res"]["equity"]))
vstats = be.trade_stats(variant["res"], dates)
vrows.loc[vrows["Period"] == "Walk-forward", list(vstats)] = list(vstats.values())
both = pd.concat([summary[summary["Strategy"] == be.WINNER["tag"]], vrows])
both.pivot_table(index="Period", columns="Strategy", values=["Total Return %", "Sharpe", "Max DD %"], sort=False)

Total Return %  \
Strategy                     C6-U96-T20-MW30-E5   
Period                                            
Walk-forward                             399.90   
Never-seen 2022-04 → 2024-09              33.20   
Last 2 years                             257.33   
Last 1 year                               52.82   

                                                                \
Strategy                     Variant: earnings_block_days=None   
Period                                                           
Walk-forward                                            414.07   
Never-seen 2022-04 → 2024-09                             33.20   
Last 2 years                                            267.46   
Last 1 year                                              54.38   

                                         Sharpe  \
Strategy                     C6-U96-T20-MW30-E5   
Period                                            
Walk-forward                               1.35   
Never-seen 2022-04 → 2024-09               0.60   
Last 2 years                               1.97   
Last 1 year                                1.39   

                                                                \
Strategy                     Variant: earnings_block_days=None   
Period                                                           
Walk-forward                                              1.37   
Never-seen 2022-04 → 2024-09                              0.60   
Last 2 years                                              2.02   
Last 1 year                                               1.41   

                                       Max DD %  \
Strategy                     C6-U96-T20-MW30-E5   
Period                                            
Walk-forward                             -30.76   
Never-seen 2022-04 → 2024-09             -29.33   
Last 2 years                             -30.76   
Last 1 year                              -23.53   

                                                                
Strategy                     Variant: earnings_block_days=None  
Period                                                          
Walk-forward                                            -32.19  
Never-seen 2022-04 → 2024-09                            -29.33  
Last 2 years                                            -32.19  
Last 1 year                                             -22.67

## 6. Save

In [8]:
if SAVE:
    out = pd.concat([summary, vrows], ignore_index=True)
    out.to_csv(be.REPORTS_DIR / "backtest_summary.csv", index=False)
    per_stock.to_csv(be.REPORTS_DIR / "backtest_per_stock.csv", index=False)
    print("Saved Reports/backtest_summary.csv and Reports/backtest_per_stock.csv")

Saved Reports/backtest_summary.csv and Reports/backtest_per_stock.csv
